# Aprendizado de Máquina — Lista prática 04

## Métodos Não Paramétricos (KNN)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

O Nadaraya--Watson cabe em duas linhas de `numpy`, e escrevê-las é a melhor forma
de entender o que ele faz. Depois disso a lista mede, sobre 300 amostras, o viés
de fronteira que a nota descreve — e o resultado tem uma reviravolta:

> **a regressão linear local reduz o viés na borda pela metade, como a teoria
> promete. Se isso a torna o método melhor ali depende da janela.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.neighbors import KNeighborsRegressor

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — Nadaraya--Watson à mão

O estimador é uma média ponderada de **todas** as observações,

$$\widehat r(x) = \sum_{i=1}^n w_i(x)\,y_i,
  \qquad w_i(x) = \frac{K\!\left(\frac{x - x_i}{h}\right)}
                       {\sum_{j=1}^n K\!\left(\frac{x - x_j}{h}\right)}.$$

Escreva os três núcleos e o estimador. Repare que a função recebe uma **grade**
de pontos de uma vez: `x0[:, None] - x_tr[None, :]` monta a matriz de todas as
diferenças, de tamanho (grade × treino).

In [ ]:
def nucleo_uniforme(u):
    return (np.abs(u) <= 1).astype(float)                      # (a)


def nucleo_gaussiano(u):
    return np.exp(-0.5 * u ** 2)                               # (b)


def nucleo_epanechnikov(u):
    return np.maximum(1 - u ** 2, 0)                           # (c)


def nadaraya_watson(x_tr, y_tr, x0, h, nucleo=nucleo_gaussiano):
    W = nucleo((x0[:, None] - x_tr[None, :]) / h)                # pesos nao normalizados
    soma = W.sum(axis=1)
    # onde nenhum ponto de treino entrou na janela, o estimador nao existe (0/0)
    return np.where(soma > 0, (W @ y_tr) / np.where(soma > 0, soma, 1), np.nan)   # (d)

A amostra é a de sempre: $r(x)=\operatorname{sen}(1{,}5x)+0{,}3x$ com ruído
$N(0;\,0{,}7^2)$, $n=50$, semente 2026.

In [ ]:
def r(x):
    return np.sin(1.5 * x) + 0.3 * x


A, B, SIGMA, N_TR = -3.0, 3.0, 0.7, 50

rng = np.random.default_rng(2026)
x_tr = rng.uniform(A, B, size=N_TR)
y_tr = r(x_tr) + rng.normal(0, SIGMA, size=N_TR)
grade = np.linspace(A, B, 300)

for nome, K in [("uniforme", nucleo_uniforme),
                ("gaussiano", nucleo_gaussiano),
                ("Epanechnikov", nucleo_epanechnikov)]:
    est = nadaraya_watson(x_tr, y_tr, grade, h=0.4, nucleo=K)
    ok = ~np.isnan(est)
    eqm = np.mean((est[ok] - r(grade[ok])) ** 2)
    print(f"{nome:14s} EQM contra r: {eqm:.4f}   pontos sem vizinho: {int((~ok).sum())}")

Deve imprimir:

```
uniforme       EQM contra r: 0.1070   pontos sem vizinho: 10
gaussiano      EQM contra r: 0.0626   pontos sem vizinho: 0
Epanechnikov   EQM contra r: 0.1318   pontos sem vizinho: 10
```

Duas leituras, e a segunda é a que importa na prática.

O **EQM** varia pouco entre os núcleos — de 0,063 a 0,132, tudo na mesma ordem
de grandeza. A escolha do núcleo quase nunca é a decisão importante; a janela $h$
é (Exercício 2).

Já a coluna da direita mostra uma diferença **qualitativa**: o uniforme e o
Epanechnikov têm suporte compacto (valem exatamente zero fora de $|u|\le 1$), e
em 10 dos 300 pontos da grade nenhuma observação caiu dentro da janela. Ali o
estimador é $0/0$ — ele simplesmente **não existe**. O núcleo gaussiano nunca tem
esse problema, porque dá peso positivo, ainda que minúsculo, a toda observação.
É por isso que ele é o padrão.

> **Sua vez.** Desenhe as três curvas ajustadas sobre a nuvem, junto com $r$
> verdadeira. Onde estão os buracos dos núcleos de suporte compacto?

---
## Exercício 2 — a janela $h$ é que decide

Agora varie $h$ e escolha o melhor por validação cruzada de 5 dobras — a
ferramenta da Aula 03 aplicada a um método que não tem `.fit()`.

In [ ]:
hs = np.array([0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8, 1.2, 2.0])
cv = skm.KFold(5, shuffle=True, random_state=2026)
eqm_cv = []

for h in hs:
    erros = []
    for i_tr, i_te in cv.split(x_tr):
        pred = nadaraya_watson(x_tr[i_tr], y_tr[i_tr], x_tr[i_te], h)   # (a) ajuste SEM a dobra
        erros.append(np.mean((y_tr[i_te] - pred) ** 2))
    eqm_cv.append(np.mean(erros))

eqm_cv = np.array(eqm_cv)
h_melhor = hs[int(np.argmin(eqm_cv))]                          # (b)

for h, e in zip(hs, eqm_cv):
    print(f"h = {h:4.2f}:  EQM (CV) {e:.4f}")
print(f"\nmelhor h por CV: {h_melhor}")

A validação cruzada escolheu $h$ **sem** conhecer $r$. Como aqui nós conhecemos,
dá para conferir se ela acertou: meça o erro contra a $r$ verdadeira na grade.

In [ ]:
est = nadaraya_watson(x_tr, y_tr, grade, h_melhor)
print(f"EQM contra r verdadeira, com h = {h_melhor}: {np.mean((est - r(grade)) ** 2):.4f}")   # (a)

A tabela deve sair assim, com o mínimo em $h=0{,}30$:

| $h$ | 0,05 | 0,10 | 0,20 | **0,30** | 0,40 | 0,60 | 0,80 | 1,20 | 2,00 |
|---|---|---|---|---|---|---|---|---|---|
| EQM (CV) | 1,0141 | 0,7906 | 0,6649 | **0,6381** | 0,6584 | 0,7161 | 0,7861 | 0,9211 | 1,0913 |

E o erro contra $r$ com esse $h$ é **0,0726**.

Compare com a variação entre núcleos do Exercício 1 (0,063 a 0,132): aqui o EQM
de CV varia de 0,64 a 1,09, quase o dobro, e nos extremos por motivos opostos —
$h=0{,}05$ é variância pura (a curva persegue cada ponto), $h=2{,}0$ é viés puro
(a curva vira quase uma reta). **A janela é o hiperparâmetro; o núcleo é
decoração.**

Note também a diferença de escala entre os dois números: 0,6381 é o erro contra
$y$ (que inclui $\sigma^2 = 0{,}49$) e 0,0726 é o erro contra $r$. A diferença,
$0{,}57$, é aproximadamente o ruído irredutível — como tem de ser.

---
## Exercício 3 — KNN na mesma amostra

O KNN fixa o **número** de vizinhos e deixa o raio variar; o Nadaraya--Watson faz
o contrário. Escolha o $k$ por CV, com as mesmas dobras, e compare os dois no
mesmo pé.

In [ ]:
X_tr = x_tr.reshape(-1, 1)
ks = np.arange(1, 26)

eqm_knn = np.array([
    -skm.cross_val_score(KNeighborsRegressor(n_neighbors=k),   # (a)
                         X_tr, y_tr, cv=cv,
                         scoring="neg_mean_squared_error").mean()
    for k in ks
])

k_melhor = ks[int(np.argmin(eqm_knn))]                         # (b)
print(f"melhor k por CV: {k_melhor}  (EQM {eqm_knn.min():.4f})")

knn = KNeighborsRegressor(n_neighbors=k_melhor).fit(X_tr, y_tr)
eqm_r = np.mean((knn.predict(grade.reshape(-1, 1)) - r(grade)) ** 2)
print(f"EQM contra r verdadeira: {eqm_r:.4f}")

Deve imprimir `melhor k por CV: 5  (EQM 0.6938)` e `EQM contra r verdadeira:
0.1236`.

Pondo lado a lado com o Exercício 2:

| | EQM por CV (contra $y$) | EQM contra $r$ |
|---|---|---|
| Nadaraya--Watson, $h=0{,}30$ | **0,6381** | **0,0726** |
| KNN, $k=5$ | 0,6938 | 0,1236 |

O NW ganha nas duas colunas, e o erro contra $r$ é 41% menor. Vale reparar que a
**ordenação é a mesma** nas duas colunas: a validação cruzada, que só viu $y$,
escolheu o método que também é melhor contra o $r$ que ela não conhece. É a
propriedade que faz a CV servir para seleção de modelos, medida aqui num caso em
que dá para conferir.

A vantagem tem explicação: com pesos que decaem suavemente, o NW produz uma curva
contínua, enquanto o KNN produz uma escada — e $r$ é suave.

---
## Exercício 4 — o viés de fronteira, medido

Aqui está o exercício central da aula, e ele **não pode ser feito numa amostra
só**: viés é uma média sobre amostras, e numa única realização o ruído domina.

Vamos comparar o Nadaraya--Watson com a **regressão linear local**, que em cada
ponto ajusta uma reta ponderada em vez de uma constante. Complete a montagem do
sistema de mínimos quadrados ponderados.

In [ ]:
def linear_local(x_tr, y_tr, x0, h):
    """Ajusta uma reta ponderada em torno de cada ponto de x0 e devolve o intercepto."""
    saida = np.empty(len(x0))
    for j, ponto in enumerate(x0):
        w = nucleo_gaussiano((ponto - x_tr) / h)
        # centramos em 'ponto', para que o intercepto SEJA a predicao ali
        Z = np.column_stack([np.ones_like(x_tr), x_tr - ponto])          # (a)
        A_ = Z.T @ (w[:, None] * Z)
        b_ = Z.T @ (w * y_tr)                                            # (b)
        saida[j] = np.linalg.solve(A_, b_)[0]                            # (c) o intercepto
    return saida

Agora repita 300 vezes: sorteie uma amostra nova, estime nos pontos de interesse
com os dois métodos, e guarde. O viés é a média das estimativas menos o valor
verdadeiro.

In [ ]:
pontos = np.array([-3.0, 0.0, 3.0])       # duas fronteiras e o centro
REPETICOES = 300

for h in (0.4, 0.8):
    rng = np.random.default_rng(2026)
    est_nw = np.zeros((REPETICOES, len(pontos)))
    est_ll = np.zeros((REPETICOES, len(pontos)))

    for b in range(REPETICOES):
        xb = rng.uniform(A, B, size=N_TR)
        yb = r(xb) + rng.normal(0, SIGMA, size=N_TR)
        est_nw[b] = nadaraya_watson(xb, yb, pontos, h)
        est_ll[b] = linear_local(xb, yb, pontos, h)

    print(f"===== h = {h} =====")
    print("  x0     metodo    vies    variancia     EQM")
    for j, x0 in enumerate(pontos):
        for nome, est in (("NW", est_nw), ("LL", est_ll)):
            vies = est[:, j].mean() - r(x0)                    # (a)
            variancia = est[:, j].var()                        # (b)
            print(f"  {x0:5.1f}   {nome}    {vies:+7.4f}   {variancia:8.4f}   "
                  f"{vies ** 2 + variancia:8.4f}")             # (c) a decomposicao da Aula 01
    print()

Deve imprimir:

```
===== h = 0.4 =====
  x0     metodo    vies    variancia     EQM
   -3.0   NW      -0.1891     0.0904     0.1262
   -3.0   LL      +0.0915     0.3754     0.3837
    0.0   NW      +0.0198     0.0675     0.0679
    0.0   LL      +0.0093     0.0457     0.0458
    3.0   NW      +0.1857     0.1076     0.1421
    3.0   LL      -0.0915     0.4219     0.4303

===== h = 0.8 =====
  x0     metodo    vies    variancia     EQM
   -3.0   NW      -0.4626     0.0559     0.2699
   -3.0   LL      +0.0848     0.1484     0.1556
    0.0   NW      +0.0166     0.0510     0.0513
    0.0   LL      +0.0089     0.0260     0.0261
    3.0   NW      +0.4573     0.0642     0.2734
    3.0   LL      -0.0769     0.1644     0.1703
```

**O viés é exatamente o que a teoria promete.** No centro ($x_0=0$) os dois são
praticamente não-viesados: 0,02 e 0,01 em $h=0{,}4$. Na fronteira o NW enviesa
seis a dez vezes mais, e o sinal é o esperado — em $x_0=-3$ ele puxa para baixo
($-0{,}19$) e em $x_0=+3$ para cima ($+0{,}19$), porque de cada lado todos os
vizinhos disponíveis estão na direção em que $r$ cresce.

**E o viés do NW cresce com a janela; o do linear local, não.** Dobrando $h$ de
0,4 para 0,8, o viés do NW na fronteira vai de 0,189 para 0,463 — mais que
dobra. O do linear local fica em 0,09. É exatamente o que a Lista Teórica 04
mostrou em conta fechada: a média dos vizinhos é puxada pela inclinação de $r$, e
ajustar uma reta desconta essa inclinação.

**Responda** na célula abaixo, como comentário: olhando a coluna do EQM, qual dos
dois métodos você usaria na fronteira? A resposta é a mesma para $h=0{,}4$ e para
$h=0{,}8$?

**A resposta muda com $h$**, e é aqui que a lista quis chegar.

Com $h=0{,}4$, o linear local tem metade do viés e mesmo assim o **EQM é três
vezes maior** (0,384 contra 0,126 em $x_0=-3$). A causa está na coluna do meio:
estimar dois parâmetros (intercepto e inclinação) com os poucos pontos que sobram
perto da borda é instável, e a variância pula de 0,09 para 0,38. O ganho de viés
não paga a conta.

Com $h=0{,}8$ o quadro se inverte: a janela maior dá pontos suficientes para o
ajuste da reta ficar estável (variância cai para 0,148), enquanto o viés do NW
explodiu. Agora o linear local ganha, 0,156 contra 0,270.

A moral não é ``o linear local é melhor na fronteira'' — é que ele **troca viés
por variância**, e se essa troca compensa depende da janela. Escolher $h$ por
validação cruzada, como no Exercício 2, resolve isso automaticamente, porque a CV
mede o EQM e não o viés.

Vale notar o que isso implica sobre a figura da nota, que compara os dois em
viés: ela está certa no que afirma, e não é a história toda. Comparar métodos por
uma das duas parcelas da decomposição, sem olhar a soma, é fácil de fazer sem
perceber.

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | o núcleo quase não muda o EQM (0,063 a 0,132), mas os de suporte compacto deixam 10 pontos sem estimativa |
| 2 | a janela muda o EQM de CV de 0,64 a 1,09; a CV escolhe $h=0{,}30$ |
| 3 | NW com $h=0{,}30$ bate KNN com $k=5$ contra $r$: 0,0726 contra 0,1236 |
| 4 | na fronteira o viés do NW é 6 a 10 vezes o do linear local — e cresce com $h$, enquanto o do linear local não |
| 4 | ainda assim, em $h=0{,}4$ o EQM do linear local na fronteira é **3× pior**, por variância |

**A seguir.** A Aula 05 explica com teoria por que todo método de vizinhança
degrada quando $p$ cresce — e o Exercício 4 já deu a pista, porque em dimensão
alta quase todo ponto é ponto de fronteira.